### Reusable Functions

In [5]:
import re
import urllib.parse
import yaml


def generate_type1_text(question_text: str, correct_answer: str) -> str:
    """Type 1: Replace parentheses （ ） or ( ) with the correct answer."""
    cleaned_question = re.sub(r"（\s*）|\(\s*\)", correct_answer, question_text)
    return cleaned_question.strip()


def generate_type2_text(question_text: str, explanation_text: str) -> str:
    """Type 2: Reconstructs full sentence using the explanation's sequence.

    Strips any surrounding whitespace between words and blanks.
    """
    # 1. Extract words inside parentheses in sequential order from explanation
    ordered_words = re.findall(r"\d+\s*\(([^)]+)\)", explanation_text)
    combined_middle = "".join(ordered_words)

    # 2. Get only the target sentence line (ignore options listed below it)
    first_line = question_text.strip().split("\n")[0]

    # 3. Replace the entire blank section and trailing/leading spaces with ordered text
    reconstructed = re.sub(
        r"\s*(____\s*)+★(\s*____)*\s*", combined_middle, first_line
    )

    return reconstructed.strip()


def build_audio_url(text: str) -> str:
    """Builds the Google Translate TTS URL from text."""
    encoded_text = urllib.parse.quote(text)
    return f"https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=ja&q={encoded_text}"


def process_quiz_yaml(yaml_content: str) -> str:
    """Main function: Parses YAML, identifies question type, and updates audio_track_url."""
    data = yaml.safe_load(yaml_content)

    for section in data:
        for q in section.get("questions", []):
            q_text = q.get("question", "")
            correct_ans = str(q.get("correct_answer", ""))
            explanation = str(q.get("explanation", ""))

            if "★" in q_text:
                full_text = generate_type2_text(q_text, explanation)
            elif "（" in q_text or "(" in q_text:
                full_text = generate_type1_text(q_text, correct_ans)
            else:
                full_text = q_text

            q["audio_track_url"] = build_audio_url(full_text)

    return yaml.dump(data, allow_unicode=True, sort_keys=False)

### Unit Tests

In [6]:
import unittest


class TestAudioUrlGenerator(unittest.TestCase):

    def test_type1_question(self):
        """Test Type 1: Parentheses replacement."""
        question = "ここに「禁煙」と（ ）。"
        correct_answer = "書かれています"
        expected = "ここに「禁煙」と書かれています。"

        result = generate_type1_text(question, correct_answer)
        self.assertEqual(result, expected)

    def test_type2_question(self):
        """Test Type 2: Full sentence reconstruction with proper space stripping."""
        question = "父に ____ ____ ★ ____ 言われた。\n\n 1: タバコを 2: 吸う 3: なと 4: 強く "
        explanation = "順序: 4 (強く) -> 1 (タバコを) -> 2 (吸う) -> 3 (なと)。★(3番目)に入るのは「3」です。"
        expected = "父に強くタバコを吸うなと言われた。"

        result = generate_type2_text(question, explanation)
        self.assertEqual(result, expected)

    def test_build_audio_url(self):
        """Test URL encoding for Japanese text."""
        text = "父に強くタバコを吸うなと言われた。"
        expected_url = "https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=ja&q=%E7%88%B6%E3%81%AB%E5%BC%B7%E3%81%8F%E3%82%BF%E3%83%90%E3%82%B3%E3%82%92%E5%90%B8%E3%81%86%E3%81%AA%E3%81%A8%E8%A8%80%E3%82%8F%E3%82%8C%E3%81%9F%E3%80%82"

        self.assertEqual(build_audio_url(text), expected_url)

    def test_process_quiz_yaml(self):
        """Test full YAML transformation with Type 2 structure."""
        raw_yaml = """
- yaml_file: test.yaml
  quiz_title: Test Quiz
  questions:
  - question_number: 20
    question: '父に ____ ____ ★ ____ 言われた。

      1: タバコを 2: 吸う 3: なと 4: 強く '
    options:
    - '1'
    - '2'
    - '3'
    - '4'
    correct_answer: '3'
    explanation: '順序: 4 (強く) -> 1 (タバコを) -> 2 (吸う) -> 3 (なと)。★(3番目)に入るのは「3」です。'
"""
        updated_yaml = process_quiz_yaml(raw_yaml)
        self.assertIn("audio_track_url:", updated_yaml)
        self.assertIn(
            "q=%E7%88%B6%E3%81%AB%E5%BC%B7%E3%81%8F%E3%82%BF%E3%83%90%E3%82%B3%E3%82%92%E5%90%B8%E3%81%86%E3%81%AA%E3%81%A8%E8%A8%80%E3%82%8F%E3%82%8C%E3%81%9F%E3%80%82",
            updated_yaml,
        )


# Run tests inside Jupyter Notebook cell
suite = unittest.TestLoader().loadTestsFromTestCase(TestAudioUrlGenerator)
unittest.TextTestRunner(verbosity=2).run(suite)

test_build_audio_url (__main__.TestAudioUrlGenerator.test_build_audio_url)
Test URL encoding for Japanese text. ... ok
test_process_quiz_yaml (__main__.TestAudioUrlGenerator.test_process_quiz_yaml)
Test full YAML transformation with Type 2 structure. ... ok
test_type1_question (__main__.TestAudioUrlGenerator.test_type1_question)
Test Type 1: Parentheses replacement. ... ok
test_type2_question (__main__.TestAudioUrlGenerator.test_type2_question)
Test Type 2: Full sentence reconstruction with proper space stripping. ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

### Update Test YAML in folder

In [14]:
import argparse
from pathlib import Path
import yaml

def process_yaml_file(input_file: Path, output_file: Path):
    """Reads raw file text to preserve exact quotes, indentations, and line breaks,

    calculates audio_track_url for each question block, and writes to output_file.
    """
    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()

    with open(input_file, "r", encoding="utf-8") as f:
        parsed_data = yaml.safe_load(f)

    if parsed_data is None:
        return

    sections = (
        parsed_data if isinstance(parsed_data, list) else [parsed_data]
    )

    parsed_questions = []
    for section in sections:
        if isinstance(section, dict):
            parsed_questions.extend(section.get("questions", []))

    if not parsed_questions:
        # If no questions exist, copy input directly to output
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(content)
        return

    lines = content.splitlines(keepends=True)
    new_lines = []

    i = 0
    q_index = 0
    while i < len(lines):
        line = lines[i]
        new_lines.append(line)

        # Remove existing audio_track_url lines if updating existing files
        if "audio_track_url:" in line:
            new_lines.pop()
            i += 1
            continue

        if (
            "- question_number:" in line or "- question:" in line
        ) and q_index < len(parsed_questions):
            q = parsed_questions[q_index]
            q_text = str(q.get("question", ""))
            correct_ans = str(q.get("correct_answer", ""))
            explanation = str(q.get("explanation", ""))

            if "★" in q_text:
                tts_text = generate_type2_text(q_text, explanation)
            elif "（" in q_text or "(" in q_text:
                tts_text = generate_type1_text(q_text, correct_ans)
            else:
                tts_text = q_text.strip().split("\n")[0]

            audio_url = build_audio_url(tts_text)

            # 2-space indentation offset relative to base item indent
            indent = line[: len(line) - len(line.lstrip())] + "  "
            q_index += 1

            # Read until explanation line
            while i + 1 < len(lines) and not (
                lines[i + 1].strip().startswith("explanation:")
            ):
                i += 1
                new_lines.append(lines[i])

            if i + 1 < len(lines) and lines[i + 1].strip().startswith(
                "explanation:"
            ):
                i += 1
                new_lines.append(lines[i])

            # Append audio_track_url with 2-space relative indent
            new_lines.append(f"{indent}audio_track_url: {audio_url}\n")

        i += 1

    # Ensure output directory exists before writing
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(new_lines)


def main(input_path: str, output_path: str):
    """Processes either a single YAML file or a directory of YAML files."""
    in_path = Path(input_path)
    out_path = Path(output_path)

    if not in_path.exists():
        print(f"Error: Input path '{input_path}' does not exist.")
        return

    # Handle directory input
    if in_path.is_dir():
        out_path.mkdir(parents=True, exist_ok=True)
        yaml_files = list(in_path.glob("*.yaml")) + list(in_path.glob("*.yml"))

        if not yaml_files:
            print(f"No YAML files found in '{input_path}'")
            return

        print(
            f"Processing {len(yaml_files)} YAML file(s) from '{input_path}' to '{output_path}'..."
        )

        for file_path in yaml_files:
            relative_file = file_path.relative_to(in_path)
            target_file = out_path / relative_file
            try:
                process_yaml_file(file_path, target_file)
                print(f"  ✓ Processed: {file_path.name} -> {target_file}")
            except Exception as e:
                print(f"  ✗ Failed to process {file_path.name}: {e}")

    # Handle single file input
    else:
        # If output path is specified as a directory or ends in trailing slash
        if out_path.is_dir() or output_path.endswith("/") or output_path.endswith("\\"):
            target_file = out_path / in_path.name
        else:
            target_file = out_path

        try:
            process_yaml_file(in_path, target_file)
            print(f"  ✓ Processed: {in_path.name} -> {target_file}")
        except Exception as e:
            print(f"  ✗ Failed to process {in_path.name}: {e}")


if __name__ == "__main__":
    main(
        input_path=r"C:\dev\temp\20260805_merge_yaml\Input",
        output_path=r"C:\dev\github_repo\skill-map\dev_helper_script\Input"
    )

Processing 14 YAML file(s) from 'C:\dev\temp\20260805_merge_yaml\Input' to 'C:\dev\github_repo\skill-map\dev_helper_script\Input'...
  ✓ Processed: jp-n3-w1-d1-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d1-test.yaml
  ✓ Processed: jp-n3-w1-d2-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d2-test.yaml
  ✓ Processed: jp-n3-w1-d3-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d3-test.yaml
  ✓ Processed: jp-n3-w1-d4-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d4-test.yaml
  ✓ Processed: jp-n3-w1-d5-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d5-test.yaml
  ✓ Processed: jp-n3-w1-d6-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d6-test.yaml
  ✓ Processed: jp-n3-w1-d7-test.yaml -> C:\dev\github_repo\skill-map\dev_helper_script\Input\jp-n3-w1-d7-test.yaml
  ✓ Processed: jp-n3-w2-d1-test.yaml -> C:\dev\github_repo\ski

### Process revision JSON

In [15]:
import argparse
import json
from pathlib import Path

def process_json_file(input_path: str, output_path: str):
    """Loads a JSON file, adds audio_track_url to each record, and saves to output_path."""
    in_file = Path(input_path)
    out_file = Path(output_path)

    if not in_file.exists():
        print(f"Error: Input file '{input_path}' does not exist.")
        return

    with open(in_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Process each record in the records list
    records = data.get("records", [])
    for record in records:
        q_text = str(record.get("question_name", ""))
        correct_ans = str(record.get("correct_answer", ""))
        explanation = str(record.get("explanation", ""))

        if "★" in q_text:
            tts_text = generate_type2_text(q_text, explanation)
        elif "（" in q_text or "(" in q_text:
            tts_text = generate_type1_text(q_text, correct_ans)
        else:
            tts_text = q_text.strip().split("\n")[0]

        record["audio_track_url"] = build_audio_url(tts_text)

    # Ensure output directory exists before saving
    out_file.parent.mkdir(parents=True, exist_ok=True)

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"✓ Processed {len(records)} record(s): '{input_path}' -> '{output_path}'")


if __name__ == "__main__":
    process_json_file(
        input_path=r"C:\dev\temp\20260805_merge_yaml\revision-data-20260805-160653.json",
        output_path=r"C:\dev\temp\20260805_merge_yaml\revision-data-20260805-160653-after.json"
    )

✓ Processed 89 record(s): 'C:\dev\temp\20260805_merge_yaml\revision-data-20260805-160653.json' -> 'C:\dev\temp\20260805_merge_yaml\revision-data-20260805-160653-after.json'
